# UE Data Mining - Projet

**Logistics**

In [ ]:
# TODO
!git clone https://github.com/Sheik-info/20262027-UE-DataMining-Project.git
%cd 20262027-UE-DataMining-Project

In [ ]:
!git pull


On explore ici les jeux de données du dossier `data/logistics/` :
- `erp_facilities.csv` : les entrepôts / sites de l'entreprise
- `erp_business_partners.csv` : les partenaires (clients, fournisseurs, transporteurs...)
- `inventory_month_end.csv` : les stocks en fin de mois par site
- `inventory_adjustments.csv` : les ajustements manuels de stock
- `goods_receipts.csv` : les réceptions de marchandises
- `shipments.csv` : les expéditions
- `purchase_order_lines.csv` : les lignes de commandes d'achat

On commence, comme pour les autres parties, par un premier chargement et une exploration rapide avant de nettoyer quoi que ce soit.

\ 1. Chargement et première exploration
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.listdir("data/logistics")

Avant de charger les fichiers, on regarde rapidement leurs dimensions et leurs colonnes.
Cela permet de vérifier la structure des données.

In [ ]:
# Je regarde rapidement les fichiers avant de commencer
for fichier in os.listdir("data/logistics"):
    df = pd.read_csv("data/logistics/" + fichier)

    print("\n==============================")
    print(fichier)
    print("Dimensions :", df.shape)
    print("Colonnes :", list(df.columns))

Les 7 fichiers sont chargés séparément afin de faciliter leur manipulation et leur analyse.

In [ ]:
df_fac = pd.read_csv("data/logistics/erp_facilities.csv")
df_bp = pd.read_csv("data/logistics/erp_business_partners.csv")
df_invm = pd.read_csv("data/logistics/inventory_month_end.csv")
df_inva = pd.read_csv("data/logistics/inventory_adjustments.csv")
df_rcv = pd.read_csv("data/logistics/goods_receipts.csv")
df_shp = pd.read_csv("data/logistics/shipments.csv")
df_po = pd.read_csv("data/logistics/purchase_order_lines.csv")

In [ ]:
dfs = {
    "facilities": df_fac,
    "business_partners": df_bp,
    "inventory_month_end": df_invm,
    "inventory_adjustments": df_inva,
    "goods_receipts": df_rcv,
    "shipments": df_shp,
    "purchase_order_lines": df_po
}

In [ ]:
df_fac.head()

Un aperçu des premières lignes est affiché afin de vérifier la structure et le contenu des données.

In [ ]:
df_po.head()

Les types des colonnes sont vérifiés afin de repérer les données qui ne sont pas dans le format attendu.


In [ ]:
for nom, df in dfs.items():
    print("\n=======================")
    print(nom)
    print(df.dtypes)

In [ ]:
for nom, df in dfs.items():
    print("\n---", nom, "---")

    for colonne in df.columns:
        nb = df[colonne].isna().sum()
        if nb > 0:
            print(colonne, ":", nb)

Certaines colonnes contenant des quantités et des prix sont actuellement au format `object`. Elles devront être converties dans un format numérique avant de poursuivre l'analyse.

 2. Nettoyage

---

#### 2.1 Lignes dupliquées

La présence de lignes dupliquées est vérifiée dans chaque fichier.

In [ ]:
for nom, df in dfs.items():
    print(nom, ":", df.duplicated().sum(), "doublon(s)")

##### Suppression des doublons


In [ ]:
df_invm = df_invm.drop_duplicates().reset_index(drop=True)
df_rcv = df_rcv.drop_duplicates().reset_index(drop=True)
df_shp = df_shp.drop_duplicates().reset_index(drop=True)
df_po = df_po.drop_duplicates().reset_index(drop=True)

dfs["inventory_month_end"] = df_invm
dfs["goods_receipts"] = df_rcv
dfs["shipments"] = df_shp
dfs["purchase_order_lines"] = df_po

for nom, df in dfs.items():
    print(nom, ":", df.shape[0], "lignes")

##### 2.2 Colonnes constantes

Les colonnes qui contiennent une seule valeur, ou qui sont entièrement vides, sont repérées. Elles n'apportent pas de variation pour l'analyse et peuvent donc être supprimées.

In [ ]:
for nom, df in dfs.items():
    cols_constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]

    if cols_constantes:
        print(nom, ":", cols_constantes)

Suppression des colonnes qui ne varient pas


In [ ]:
df_fac = df_fac.drop(columns=["facility_status"])

df_rcv = df_rcv.drop(columns=["purchase_order_line"])
df_po = df_po.drop(columns=["purchase_order_line"])

df_shp = df_shp.drop(columns=["shipment_leg"])

dfs["facilities"] = df_fac
dfs["goods_receipts"] = df_rcv
dfs["purchase_order_lines"] = df_po
dfs["shipments"] = df_shp

#### 2.3 Nettoyage des colonnes numériques — inventory_month_end

Les colonnes `opening_stock`, `received_quantity`, `outgoing_quantity`, `documented_waste` et `closing_stock` sont actuellement de type `object` alors qu'elles devraient être numériques. Deux problèmes de format sont en cause :

- certaines valeurs utilisent une virgule comme séparateur décimal (ex : `"5,42"`) au lieu d'un point ;
- certaines valeurs contiennent l'unité collée au nombre (ex : `"14.55 sealed_lot"`). Cette unité est redondante avec la colonne `unit` déjà présente, elle peut donc être retirée sans perte d'information.

Une fonction de nettoyage est appliquée à ces colonnes.

In [ ]:
import re

def nettoyer_colonne_numerique(serie):
    s = serie.astype(str).str.strip()
    s = s.str.replace(',', '.', regex=False)
    s = s.str.extract(r'(-?\d+\.?\d*)')[0]
    return pd.to_numeric(s, errors='coerce')

colonnes_a_nettoyer = ['opening_stock', 'received_quantity', 'outgoing_quantity',
                        'documented_waste', 'closing_stock']

for col in colonnes_a_nettoyer:
    df_invm[col] = nettoyer_colonne_numerique(df_invm[col])

df_invm.dtypes

La colonne `inventory_status` présente aussi une incohérence de casse (`"closed"` / `"CLOSED"`, `"provisional"` / `"PROVISIONAL"`). Elle est uniformisée en minuscules.

In [ ]:
df_invm['inventory_status'] = df_invm['inventory_status'].str.lower()
df_invm['inventory_status'].value_counts()

##### Vérification de cohérence comptable

Le stock de fin de mois (`closing_stock`) devrait théoriquement correspondre à :
`opening_stock + received_quantity − outgoing_quantity − documented_internal_use − documented_waste`.

Un écart entre valeur déclarée et valeur recalculée est ajouté comme nouvelle colonne, afin d'être analysé plus tard plutôt que d'être corrigé arbitrairement.

In [ ]:
dfs["inventory_month_end"] = df_invm

In [ ]:
df_invm.dtypes

In [ ]:
df_invm.head()

#### 2.4 Nettoyage des colonnes numériques — goods_receipts

La colonne `received_quantity` présente les mêmes problèmes de format que `inventory_month_end` : virgule décimale et unité collée à la valeur (redondante avec `received_unit`). La même fonction de nettoyage est réutilisée.

In [ ]:
df_rcv['received_quantity'] = nettoyer_colonne_numerique(df_rcv['received_quantity'])
df_rcv['received_quantity'].describe()

La colonne `receipt_status` présente une incohérence de casse (`"posted"` / `"POSTED"`), uniformisée en minuscules.

In [ ]:
df_rcv['receipt_status'] = df_rcv['receipt_status'].str.lower()
df_rcv['receipt_status'].value_counts()

4 lignes présentent une valeur manquante pour `inspection_status`, toutes avec un `receipt_status` égal à `"posted"`. Plutôt que de supposer une valeur, ces cas sont rendus explicites avec la catégorie `"non_renseigne"`.

In [ ]:
df_rcv['inspection_status'] = df_rcv['inspection_status'].fillna('non_renseigne')
df_rcv['inspection_status'].value_counts()

In [ ]:
dfs["goods_receipts"] = df_rcv

In [ ]:
df_rcv.dtypes

In [ ]:
df_rcv.head()

#### 2.5 Nettoyage des colonnes numériques — purchase_order_lines

Les colonnes `ordered_quantity` (unité collée) et `unit_price` (virgule décimale) présentent les mêmes problèmes de format que les fichiers précédents.

In [ ]:
df_po['ordered_quantity'] = nettoyer_colonne_numerique(df_po['ordered_quantity'])
df_po['unit_price'] = nettoyer_colonne_numerique(df_po['unit_price'])
df_po[['ordered_quantity', 'unit_price']].describe()

La colonne `item_category` présente des doublons de casse (ex : `"monitoring_equipment"` / `"MONITORING_EQUIPMENT"`), uniformisés en minuscules.

In [ ]:
df_po['item_category'] = df_po['item_category'].str.lower()
df_po['item_category'].value_counts()

3 lignes présentent une valeur manquante pour `declared_cost_center`, sans motif apparent. Comme pour `inspection_status`, ces cas sont rendus explicites avec la catégorie `"non_renseigne"` plutôt que de supposer une valeur.

In [ ]:
df_po['declared_cost_center'] = df_po['declared_cost_center'].fillna('non_renseigne')
df_po['declared_cost_center'].value_counts()

**Point de vigilance pour l'analyse** : `unit_price` est exprimé dans 5 devises différentes (VLN, AVL, TMB, NDH, KRN). Toute comparaison  de prix entre commandes nécessitera une standardisation (a faire plus tard)

In [ ]:
dfs["purchase_order_lines"] = df_po

In [ ]:
df_po.head()

#### 2.6 Nettoyage des colonnes — inventory_adjustments

`adjustment_quantity` présente une unité collée sur une ligne (redondante avec `unit`), retirée avec la même fonction que précédemment.

In [ ]:
df_inva['adjustment_quantity'] = nettoyer_colonne_numerique(df_inva['adjustment_quantity'])
df_inva['adjustment_quantity'].describe()

`adjustment_reason` présente des doublons de casse, uniformisés en minuscules.

In [ ]:
df_inva['adjustment_reason'] = df_inva['adjustment_reason'].str.lower()
df_inva['adjustment_reason'].value_counts()

In [ ]:
df_inva['approved_by_role'] = df_inva['approved_by_role'].fillna('non_renseigne')

In [ ]:
dfs["inventory_adjustments"] = df_inva

In [ ]:
df_inva.head()

#### 2.7 Nettoyage — erp_facilities
 Ce fichier ne présente pas de problème de format numérique ni de casse. Seule la colonne storage_capabilities contient une valeur manquante



In [ ]:
df_fac['storage_capabilities'] = df_fac['storage_capabilities'].fillna('non_renseigne')
df_fac['storage_capabilities'].value_counts()

mise à jour du dic

In [ ]:
dfs["facilities"] = df_fac
df_fac.head()

###2.8 Nettoyage — erp_business_partners
partner_role présente une incohérence de casse ("Supplier" / "supplier"), comme observé précédemment pour item_category dans purchase_order_lines. Elle est uniformisée en minuscules.

In [ ]:
df_bp['partner_role'] = df_bp['partner_role'].str.lower()
df_bp['partner_role'].value_counts()

external_registration_number (14 valeurs manquantes) et billing_region (1 valeur manquante) ne présentent pas de motif apparent expliquant leur absence. Ces cas sont rendus explicites avec la catégorie "non_renseigne", dans la continuité de la démarche adoptée pour les autres fichiers

In [ ]:
df_bp['external_registration_number'] = df_bp['external_registration_number'].fillna('non_renseigne')
df_bp['billing_region'] = df_bp['billing_region'].fillna('non_renseigne')

Point de vigilance pour l'analyse : la colonne shipping_region mélange deux granularités géographiques différentes (ex : "Valenne Central District" face à "Valenne Republic"). Cette hétérogénéité n'est pas corrigée ici mais devra être prise en compte lors d'agrégations géographiques (à faire lors de l'analyse)

In [ ]:
dfs["business_partners"] = df_bp
df_bp.head()

###2.9 Nettoyage — shipments

Les doublons et la colonne constante (shipment_leg) ont déjà été traités dans les étapes 2.1 et 2.2. Il reste des incohérences de casse sur deux colonnes catégorielles, un problème d'espaces multiples sur une troisième, et des valeurs manquantes sur arrival_time.

In [ ]:
df_shp['temperature_control_class'] = df_shp['temperature_control_class'].str.lower()
df_shp['shipment_status'] = df_shp['shipment_status'].str.lower()

df_shp['temperature_control_class'].value_counts()

In [ ]:
df_shp['shipment_status'].value_counts()

declared_purpose présente un problème différent des cas précédents : certaines valeurs contiennent des espaces multiples ou en début de chaîne (ex : " laboratory delivery" au lieu de "laboratory delivery"), ce qui crée artificiellement de fausses catégories distinctes. Une fonction dédiée est utilisée pour normaliser les espaces, sur le même principe que nettoyer_colonne_numerique.

In [ ]:
def nettoyer_espaces(serie):
    s = serie.astype(str).str.strip()
    s = s.str.replace(r'\s+', ' ', regex=True)
    return s

df_shp['declared_purpose'] = nettoyer_espaces(df_shp['declared_purpose'])
df_shp['declared_purpose'].value_counts()

vérification, pas de modification

In [ ]:
df_shp[df_shp['arrival_time'].isna()]['shipment_status'].value_counts()

mise à jour du dict

In [ ]:
dfs["shipments"] = df_shp
df_shp.head()

In [ ]:
#simple verif
for nom, df in dfs.items():
    print(nom, ":", df.shape)

#### Bilan nettoyage — logistics

On a nettoyé les 7 fichiers un par un en suivant à chaque fois la même logique :

- on a supprimé les doublons (lignes identiques) et les colonnes qui ne varient jamais (ex : `facility_status`, `shipment_leg`), elles ne servent à rien pour l'analyse
- on a corrigé les colonnes numériques qui étaient stockées en texte (virgule au lieu de point, unité collée genre "12kg"), avec la fonction `nettoyer_colonne_numerique`
- on a uniformisé les catégories qui avaient des soucis de casse (ex : "Supplier" vs "supplier", "DELIVERED" vs "delivered"), en mettant tout en minuscules
- pour les valeurs manquantes, on a essayé à chaque fois de comprendre pourquoi elles étaient manquantes avant de faire quoi que ce soit. Quand on n'avait pas d'explication, on a préféré les marquer clairement comme `"non_renseigne"` plutôt que d'inventer une valeur. Et quand l'absence avait un sens (ex : pas de `arrival_time` parce que le colis est encore `in_transit`), on l'a laissée telle quelle

**Trucs à garder en tête pour la suite** :
- `unit_price` est dans 5 devises différentes, donc pas comparable tel quel entre commandes
- `shipping_region` mélange des noms de région et des noms de pays, à uniformiser avant de faire des stats dessus

**Ce qu'il reste à faire** : on n'a pas encore regardé les valeurs aberrantes (outliers) dans les colonnes numériques, c'est la prochaine étape avant de dire que le nettoyage logistics est vraiment terminé.